In [11]:
import os
from langchain_core.documents import Document

In [12]:
os.makedirs("data/text_files", exist_ok = True)
os.makedirs("data/pdfs", exist_ok = True)

In [14]:
from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader, TextLoader, CSVLoader

# pdfDoc = PyMuPDFLoader('data/pdfs/Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf')
# docs = pdfDoc.load()
# len(docs)

doc = DirectoryLoader(
    "data/pdfs",
    loader_cls= PyMuPDFLoader,
    glob = "**/*.pdf"
)

documents = doc.load()
documents

[Document(metadata={'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-06T16:40:24+00:00', 'source': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'file_path': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'total_pages': 578, 'format': 'PDF 1.6', 'title': 'Machine Learning Engineering in Action', 'author': 'Ben Wilson', 'subject': '', 'keywords': '', 'moddate': 'D:20220322233007', 'trapped': '', 'modDate': 'D:20220322233007', 'creationDate': 'D:20220306164024Z', 'page': 0}, page_content='M A N N I N G\nBen Wilson\nIN ACTION'),
 Document(metadata={'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF E

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

def process_all_pdf(directory):
    all_documents = []
    dir = Path(directory)

    # pdf_files = list(pdf_dir.glob('**/*.pdf')) 
    files = list(dir.glob('**/*.*')) 

    print(f'Found {len(files)} files in the directory.')

    loader_map = {
        '**/*.pdf' : PyMuPDFLoader,
        '**/*.txt' : TextLoader,
        '**/*.csv' : CSVLoader
    }

    # for pdf_file in pdf_files:
    #     # loader = PyMuPDFLoader(pdf_file)
    #     loader = DirectoryLoader(
    #         pdf_files
    #     )
    #     documents = loader.load()

    for pattern, loader_cls in loader_map.items():
        loader = DirectoryLoader(
            directory,
            glob=pattern,
            loader_cls = loader_cls,
            use_multithreading=True,
            max_concurrency=4,
            silent_errors=True # Skips corrupted files instead of reading.
        )

        all_documents.extend(loader.load())

    for doc in all_documents:
        # path = doc.metadata.get('source', 'unknown')
        # print(path.split('\\')[-1][-3:])
        # doc.metadata['source'] = path.split('\\')[-1]
        # doc.metadata["file_type"] = 'pdf'
        p = Path(doc.metadata['source'])
        # path_str = doc.metadata.get("source", "")
        # path_obj = Path(path_str)
        
        # .suffix gives you '.pdf' (includes the dot)
        # .stem gives you 'filename' (no extension)
        doc.metadata["filetype"] = p.suffix[1:].lower()
        doc.metadata["filename"] = p.name
    
    # all_documents.extend(documents)
    print(f'Total Documents Loaded: {len(documents)}')

    return all_documents

all_pdfs_documents = process_all_pdf('./data')

Found 4 files in the directory.
Total Documents Loaded: 2307


In [23]:
import tiktoken
# Create chunks.
# 1. Initialize the tokenizer for your specific model
# 'cl100k_base' is used for GPT-3.5, GPT-4, and GPT-4o
tokenizer = tiktoken.get_encoding("cl100k_base")

# 2. Define a function that takes text and returns the token count
def tiktoken_len(text):
    tokens = tokenizer.encode(text, disallowed_special=())
    return len(tokens)

def split_documents(documents, chunk_size = 250, chunk_overlap = 30):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_doc = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_doc)} chunks.")

    #Show example of chuncks.
    if split_doc:
        print(f'\nExample chunks:')
        print(f'Content: {split_doc[0].page_content[:200]}...')
        print(f'Metadata: {split_doc[0].metadata}')

    return split_doc

In [24]:
chunks = split_documents(all_pdfs_documents)
chunks

Split 2307 documents into 19117 chunks.

Example chunks:
Content: M A N N I N G
Ben Wilson
IN ACTION...
Metadata: {'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-06T16:40:24+00:00', 'source': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'file_path': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'total_pages': 578, 'format': 'PDF 1.6', 'title': 'Machine Learning Engineering in Action', 'author': 'Ben Wilson', 'subject': '', 'keywords': '', 'moddate': 'D:20220322233007', 'trapped': '', 'modDate': 'D:20220322233007', 'creationDate': 'D:20220306164024Z', 'page': 0, 'filetype': 'pdf', 'filename': 'Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-06T16:40:24+00:00', 'source': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'file_path': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'total_pages': 578, 'format': 'PDF 1.6', 'title': 'Machine Learning Engineering in Action', 'author': 'Ben Wilson', 'subject': '', 'keywords': '', 'moddate': 'D:20220322233007', 'trapped': '', 'modDate': 'D:20220322233007', 'creationDate': 'D:20220306164024Z', 'page': 0, 'filetype': 'pdf', 'filename': 'Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf'}, page_content='M A N N I N G\nBen Wilson\nIN ACTION'),
 Document(metadata={'producer': 'Acrobat Distiller 20.0 (Wind

# Embedding and Vector Store DB

In [25]:
    import numpy as np
    from sentence_transformers import SentenceTransformer
    import chromadb
    from chromadb.config import Settings
    import uuid
    from typing import List, Dict, Any, Tuple
    from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        # Use huggingface model name for sentence embedding
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model is loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except: 
            print(f'Error loading model {self.model_name}: {e}')
            raise
    
    def generate_embeddings(self, documents):
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(documents)} docuemnts")
        embeddings = self.model.encode(documents, show_progress_bar=True)
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name sentence-transformers/all-MiniLM-L6-v2. Creating a new one with mean pooling.
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


NameError: name 'e' is not defined